### Connecting to ADLS

In [0]:
from pyspark.sql.functions import col, md5, cast, concat_ws, hex, expr, lit

In [0]:
%run ../utils/adlsConfig

### Data Ingestion


#### 1. JDBC Functions

##### Historical JDBC

In [0]:
# Reads table from source and Returns a dataframe, count
def readSqlserverHistorical(config, jcConfig):
  if config["dateColumnName"]!=None:
    if not config['selectColList']:
      query = f"""SELECT * FROM {sourceObjectName} WHERE {config["dateColumnName"]} < convert(date,'{jcConfig["currentRunDate"]}')"""
    else:
      query = f"""SELECT {(config['selectColList'])} FROM {sourceObjectName} WHERE {config["dateColumnName"]} < convert(date,'{jcConfig["currentRunDate"]}')"""   
      #Enter the column names in config['selectColList'] as <x,y,z> if x,y,z are column names

  else:
    if not config['selectColList']:
      query = f"""SELECT * FROM {sourceObjectName}"""
    else:
      query = f"""SELECT {(config['selectColList'])} FROM {sourceObjectName}"""   

  logger.info("Reading the Historical Data from the source",extra=properties)
  
  tableSrc =  spark.read\
        .format("com.microsoft.sqlserver.jdbc.spark")\
        .option("driver", driver)\
        .option("url", jdbcUrl)\
        .option("query", query)\
        .option("user", username)\
        .option("password", password)\
        .option("trustServerCertificate", "true")\
        .load()
  tableSorted=tableSrc.select(sorted(tableSrc.columns,reverse=False))


  return tableSorted,tableSorted.count()

In [0]:
# Reads table from source and Returns a dataframe, count
def readSqlserverHistoricalOptimized(config, jcConfig):
  try:
    jsonValues = eval(config['customConfigJson'])
    partitionColumn=jsonValues['partitionColumn']
    numPartitions=jsonValues['numPartitions']
    logger.info("Calculating min and max values",extra=properties)
    query = f"""SELECT min({partitionColumn}) as min,max({partitionColumn}) as max FROM {sourceObjectName}"""
    # Load data from SQL Server into a tableSrc
    df=spark.read\
  .format("com.microsoft.sqlserver.jdbc.spark")\
  .option("driver", driver)\
  .option("url", jdbcUrl)\
  .option("query", query)\
  .option("user", username)\
  .option("password", password)\
  .option("trustServerCertificate", "true")\
  .load()

    min=df.first()[0]
    max=df.first()[1]
    
    logger.info("Reading the Historical Data from the source in the Optimized way",extra=properties)
    
    # Load data from SQL Server into a tableSrc
    tableSrc =  spark.read\
    .format("com.microsoft.sqlserver.jdbc.spark")\
    .option("driver", driver)\
    .option("url", jdbcUrl)\
    .option("dbtable", sourceObjectName)\
    .option("numPartitions", int(numPartitions))\
    .option("lowerBound", min)\
    .option("upperBound", max)\
    .option("partitionColumn",str(partitionColumn))\
    .option("fetchsize", 1000)\
    .option("user", username)\
    .option("password", password)\
    .option("trustServerCertificate", "true")\
    .load()
    tableSorted=tableSrc.select(sorted(tableSrc.columns,reverse=False))
    tableFiltered=tableSorted.where(tableSorted.dateLastSync< jcConfig["currentRunDate"].strftime('%Y-%m-%d %H:%M:%S'))

  except Exception as e:        
    logger.exception(f"Invalid customConfigJson Parameters Provided in the Configuration File,Plase check and provide in correct format {e}",extra = properties)
    raise e
  return tableFiltered,tableFiltered.count()


##### Incremental/Custom JDBC Function

In [0]:
def readSqlserverCustom(config, jcConfig):
  if config["dateColumnName"]!=None:
    query = f"""SELECT {'{columns}'} FROM {config["sourceObjectName"]} WHERE {config["dateColumnName"]} >= convert(date,'{jcConfig["lastRunDate"]}') AND {config["dateColumnName"]} < convert(date,'{jcConfig["currentRunDate"]}')"""
  else:
    query = f"""SELECT {'{columns}'} FROM {config["sourceObjectName"]}"""
    
  if not config['selectColList']: 
    query = query.format(columns="*")
  else:
    query = query.format(columns=(config['selectColList']))
  logger.info("Reading the Incremental/Custom Data from the source",extra=properties)
  tableSrc =  spark.read \
        .format("com.microsoft.sqlserver.jdbc.spark")\
        .option("driver", driver)\
        .option("url", jdbcUrl)\
        .option("query", query)\
        .option("user", username)\
        .option("password", password)\
        .option("trustServerCertificate", "true")\
        .load()
  tableSorted=tableSrc.select(sorted(tableSrc.columns,reverse=False))
  return tableSrc, tableSrc.count()


#### 2. ADLS Functions

##### Historical ADLS

In [0]:
# ADLSHistorical
def readAdlsHistorical(config, jcConfig):
  try:
    if config["rawObjectFormat"].lower() == "csv":
      #Definining Source File path
      filePath = f"{baseAdlsPath}{sourceFormatPath}.csv"
      #Reading source file csv
      if config['customConfigJson']:
        tableSrc = spark.read.format("csv") \
        .option("header", config['customConfigJson']['header']) \
        .option("delimiter", config['customConfigJson']['delimiter']) \
        .option("inferSchema", True) \
        .load(filePath)
      else:
        tableSrc = spark.read.format("csv") \
        .option("header", True) \
        .option("inferSchema", True) \
        .load(filePath)
      
      # Add a log message to indicate that the data has been read successfully
      logger.info(f"{config['sourceObjectName']}.csv file read successfully.",extra = properties)
      
    else:
      # Add a log message to indicate that there's an issue with the rawObjectFormat configuration
      logger.error("Check the rawObjectFormat, that is different",extra = properties)
      
  except Exception as e:
    # Add an exception block to handle any errors that may occur during the data reading process
    logger.exception(f"Error reading {config['sourceObjectName']}.csv file.",extra = properties)
    tableSrc = None
    raise e
  
  return tableSrc, tableSrc.count()


##### Incremental/Custom ADLS

In [0]:
# ADLSIncremental
def readAdlsCustom(config, jcConfig):
   
    try:
        if config["rawObjectFormat"].lower() == "csv":
            #Defining Source File path
            #filePath = f"/mnt/{storage_account_name}/{container_name}/{config['sourcePath']}/{config['sourceObjectName']}.csv"
            filePath=f"{baseAdlsPath}{sourceFormatPath}.csv"

            #Reading source file csv
            if config['customConfigJson']:
                tableSrc = spark.read.format("csv") \
                    .option("header", config['customConfigJson']['header']) \
                    .option("delimiter", config['customConfigJson']['delimiter']) \
                    .option("inferSchema", True) \
                    .load(filePath)
                tableDfFiltered = tableSrc.filter((col(config["dateColumnName"]) >= jcConfig["lastRunDate"]) & (col(config["dateColumnName"]) < jcConfig["currentRunDate"]))
            else:
                tableSrc = spark.read.format("csv") \
                    .option("header", True) \
                    .option("inferSchema", True) \
                    .load(filePath)
                tableDfFiltered = tableSrc.filter((col(config["dateColumnName"]) >= jcConfig["lastRunDate"]) & (col(config["dateColumnName"]) < jcConfig["currentRunDate"]))
        else:
            raise ValueError("Invalid rawObjectFormat. Expected 'csv'.")

        # Log the number of rows in the filtered DataFrame.
        logger.info(f"Filtered DataFrame has {tableDfFiltered.count()} rows.",extra = properties)

        return tableDfFiltered, tableDfFiltered.count()

    except Exception as e:
        # Log the exception.
        logger.exception(f"An error occurred while reading the CSV file: {e}",extra = properties)
        raise e

In [0]:
readAdlsIncremental = readAdlsCustom

In [0]:
readSqlserverIncremental = readSqlserverCustom

#### 3. md5Checksum Function

In [0]:
#     Calculates the MD5 checksum for each column of a Spark DataFrame and concatenates them into a single string.
#     Args: tableDf : The Spark DataFrame to calculate the MD5 checksum for.
#     Returns:A new Spark DataFrame with an additional column "md5CheckSum" that contains the concatenated MD5 checksums.


def generateMd5(tableDfChck):
    
    try:
        # Concatenate the MD5 checksums into a single string with "~" as the separator
         tableDfOut = tableDfChck.withColumn("dwMd5CheckSum", concat_ws("~", *[md5(col(c).cast("string")).alias(c) for c in tableDfChck.columns]))
        
    except Exception as e:        
        logger.error(f"MD5 checksum calculation failed: {e}",extra = properties)
        raise e
        
    return tableDfOut
  

#### 4. Adhoc Run Functions

In [0]:
def adhocSourceToRaw(source):
  try:
    dbutils.notebook.run("../scripts/sourceToRaw", timeout_seconds = 1800, arguments = {"sourceObjectName": source.split("~")[1], "sourceSystem": source.split("~")[0]})
    return True
  except Exception as e:
    raise e

In [0]:
def adhocRawToCuration(source):
  try:
    dbutils.notebook.run("../scripts/rawToCuration", timeout_seconds = 1800, arguments = {"sourceObjectName": source.split("~")[1], "sourceSystem": source.split("~")[0]})
    return True
  except Exception as e:
    raise e

In [0]:
def adhocDqExecution(source):
  try:
    dbutils.notebook.run("../dataQuality/dqExecution", timeout_seconds = 1800, arguments = {"sourceObjectName": source.split("~")[1], "sourceSystem": source.split("~")[0]})
    return True
  except Exception as e:
    raise e

#### 5. Email Alert Function

In [0]:
def sendEmailAlert(senderEmail, recipientEmail,emailSubject,html=None):
    try:
        # Set up an SMTP connection
        server = smtplib.SMTP_SSL('smtp.sendgrid.net', 465)
        server.ehlo()

        # Log in using the sender's email and password
        server.login(apiKey, apiSecret)

        # Compose the email message
        msg = MIMEMultipart()
        msg['From'] = senderEmail
        msg['To'] = recipientEmail
        msg['Subject'] = emailSubject
        
        today = date.today()
        formattedDate = today.strftime("%B %d, %Y")
        
        message = """
        <html>
            <body>
                <p>Hi,</p>
                <p>Aggregate DQ report for entire batch run as of {date} :</p>
                {table}
                <p>Regards,<br>Ace Data Platform</p>
            </body>
        </html>
        """.format(date=formattedDate,table=htmlTable)

        # Attach the HTML message to the email
        html = MIMEText(message, 'html')
        msg.attach(html)
                
      
        # Send the email
        server.sendmail(senderEmail, recipientEmail, msg.as_string())

        # Close the SMTP connection
        server.quit()
        logger.info("Email alert sent successfully!",extra=properties)     
    except Exception as e:
        logger.error("Error sending email alert:",extra=properties)
      

####    6. Pipeline Audit Table Update Raw

In [0]:
def updateAuditTable(auditStatus, errorLog):
    try:
        if auditStatus.lower().strip() == "Successful".lower().strip():
            stageObjectName
            stageStartTs
            stageName.lower().strip()
            int(stageOrder)
            loadType = workloadType if workloadType != None else None
            stageStatus = auditStatus.lower()
            errorLog = "Null"

            logger.info(
                "Updating pipeline audit table with status Successful", extra=properties
            )
            spark.sql(
                f"""INSERT INTO {pipelineAuditTable}(sourceSystem,sourceObjectName,stageName, stageOrder, loadType, adbPipelineID, stageObjectName, stageStartTs, stageEndTs,srcStartTs, srcEndTs,stageStatus, errorLog)
        VALUES ("{sourceSystem}","{sourceObjectName}","{stageName}", {stageOrder}, "{loadType}", "{adbpipelineId}", "{stageObjectName}", "{stageStartTs}", current_timestamp, try_cast("{srcStartTs}" as timestamp),try_cast("{srcEndTs}" as timestamp), "{stageStatus}", {errorLog})"""
            )

        elif auditStatus.lower().strip() == "Failed".lower().strip():
            stageObjectName
            stageStartTs
            stageName.lower().strip()
            int(stageOrder)
            loadType = workloadType if workloadType != None else None
            stageStatus = auditStatus.lower()

            str(errorLog)

            logger.info(
                "Updating pipeline audit table with status Failed", extra=properties
            )
            spark.sql(
                f"""INSERT INTO {pipelineAuditTable}(sourceSystem,sourceObjectName,stageName, stageOrder, loadType, adbPipelineID, stageObjectName, stageStartTs, stageEndTs,srcStartTs, srcEndTs,stageStatus, errorLog)
        VALUES ("{sourceSystem}","{sourceObjectName}","{stageName}", {stageOrder}, "{loadType}", "{adbpipelineId}", "{stageObjectName}", "{stageStartTs}", current_timestamp,  try_cast("null" as timestamp), try_cast("null" as timestamp), "{stageStatus}", "{errorLog}") """
            )

    except Exception as e:
        logger.exception(
            "An error occured in audit table update function", extra=properties
        )
        raise e

####    7. Function to check if a Path Exists or not

In [0]:
def filesCheck(path):
  if path != None:
    try:
      dbutils.fs.ls(path)
      return True
    except:
      return False
  else:
    return False

#### 8. Function to check if delta table exist or not

In [0]:
def deltaTableCheck(path):
  status = DeltaTable.isDeltaTable(spark, path)
  return status

#### 9. Function to create delta table

In [0]:
def createDeltaTable(DbName,sourceSystem,sourceObjectName,path,fileType):
  spark.sql(f"""CREATE TABLE IF NOT EXISTS {DbName}.{sourceSystem}{sourceObjectName} USING {fileType} LOCATION '{path}'""")

####    10. Archival Process 

In [0]:
def TempArchivalProcess(config,path,tempPath,srcFormat):
    logger.info(f"Starting to move the Parquet files to Temp Location",extra = properties)
    sourceObjectNameFolder = dbutils.fs.ls(path)
    if (f for f in sourceObjectNameFolder):
        filesToMove = [file.path for file in sourceObjectNameFolder if file.name.endswith('.'+srcFormat)]
        for file in filesToMove:    
            dbutils.fs.mkdirs(tempPath)
            dbutils.fs.mv(file, tempPath, True)
            logger.info(f"Successfully Moved {file} to {tempPath}",extra = properties)
            
            
        filesToDel = [file.path for file in dbutils.fs.ls(parquetPath) if file.name.endswith('_SUCCESS')]
        for file1 in filesToDel:
            dbutils.fs.rm(file1)
    else:
        logger.exception("Error occurred while moving files from Parquet to Temp Archival Path",extra = properties)
        raise ValueError("Error occurred while moving files from Parquet to Temp Archival Path")


In [0]:
from datetime import datetime
import datetime as dt
def FinalArchivalProcess(tempPath,archivalPath,parquetPath,archivalStatus):
    if archivalStatus==True:
        dt1=dt.datetime.now()
        subfolderName=dt.datetime.strftime(dt1-timedelta(days =1),"%Y-%m-%d")
        arSubFolderPath = f"{archivalPath}/{subfolderName}/"
        dbutils.fs.mkdirs(arSubFolderPath)
        logger.info(f"Moving the Parquet Files from Temp to Archival Path as the Ingestion succeeded",extra = properties)
        if (f for f in dbutils.fs.ls(tempPath)):
            filesToMove = [file.path for file in dbutils.fs.ls(tempPath)]
            for file in filesToMove:
                dbutils.fs.mv(file, f"{arSubFolderPath}", True)
                logger.info(f"Successfully Moved {file} to {archivalPath}",extra = properties)
            dbutils.fs.rm(tempPath)

        else:
            logger.exception("Error occurred while moving files from Temp to Archival Path",extra = properties)
            raise ValueError("Error occurred while moving files from Temp to Archival Path")
            
    elif archivalStatus=='Empty':
      logger.info(f"Moving the Parquet Files from Temp to Parquet Path as the Data Read from Source is Empty",extra = properties)
      if (f for f in dbutils.fs.ls(tempPath)):
        filesToMove = [file.path for file in dbutils.fs.ls(tempPath)]
        for file in filesToMove:
          dbutils.fs.mv(file, f"{parquetPath}", True)
        dbutils.fs.rm(tempPath)
        logger.info("Ingestion did not happen for this Load as the data read from source was 0,Files have been moved from Temp to Parquet Path",extra = properties)
        dbutils.notebook.exit(f"Notebook {notebookName} exited as the data read from source was 0 for {config['sourceObjectName']},Cannot proceed with the Load") 
      else:
        logger.exception("Error occurred while moving files from Temp to Parquet Path",extra = properties)
        raise ValueError("Error occurred while moving files from Temp to Parquet Path")

            
            
    else:
        logger.info(f"Moving the Parquet Files from Temp to Parquet Path as the Ingestion Failed",extra = properties)
        if (f for f in dbutils.fs.ls(tempPath)):
            filesToMove = [file.path for file in dbutils.fs.ls(tempPath)]
            for file in filesToMove:
                dbutils.fs.mv(file, f"{parquetPath}", True)
            dbutils.fs.rm(tempPath)
            logger.exception("Ingestion has Failed for this Load,Files have been moved from Temp to Parquet Path",extra = properties)
            raise ValueError("Ingestion has Failed for this Load,Files have been moved from Temp to Parquet Path")
        else:
          logger.exception("Error occurred while moving files from Temp to Parquet Path",extra = properties)
          raise ValueError("Error occurred while moving files from Temp to Parquet Path")




####    11. Data Ingestion Process

In [0]:
def dataIngestionProcess(tableSrc):
    try:
        logger.info("Calling generateMd5 method to add dwmd5CheckSum column ",extra=properties)
        tableDfChcksum = generateMd5(tableSrc)
        # If 'isDeleted' column exists, set the 'cdcIndicator' column to 'A' for rows where 'isDeleted' is '0' and 'D' for other rows
        if 'isDeleted' in tableDfChcksum.columns:
            logger.info("Adding audit column cdcIndicator",extra=properties)
            tableDfChcksum=tableDfChcksum.withColumn("isDeleted", when(lower(tableDfChcksum['isDeleted']) == 'true',1).when(lower(tableDfChcksum['isDeleted']) == 'false',0))
            tableDfFinal = tableDfChcksum.withColumn("cdcIndicator", when(tableDfChcksum['isDeleted'] =='0','A').otherwise('D'))
        else:
            # If 'isDeleted' column doesn't exist, set the 'cdcIndicator' column to 'A' for all rows
            logger.info("Adding audit column cdcIndicator with 'A'",extra=properties)
            tableDfFinal = tableDfChcksum.withColumn("cdcIndicator", lit('A'))
            # Add the current timestamp as 'dwModifiedDate' column
        logger.info("Adding dwModifiedDate column with current timestamp",extra=properties)
        tableDfWrite = tableDfFinal.withColumn("dwModifiedDate", current_timestamp())
        logger.info(f"tableDf - dataframe, which has read the {sourceObjectName} from source Location",extra=properties)

        logger.info(f"Initializing process of writing table into {config['rawObjectFormat']} at its path",extra=properties)

        tableDfWrite.write.format("parquet").mode("append").save(parquetPath)
        logger.info(f"Initializing process of writing data into PARQUET Table",extra=properties)
        
        createDeltaTable(rawDbName,sourceSystem,objectName,parquetPath,'PARQUET')
        dfTgt=spark.read.format('parquet').load(parquetPath)
        countWrite=dfTgt.count()
        spark.sql("""REFRESH TABLE {}.{}{}""".format(rawDbName,sourceSystem, objectName))

        
    except Exception as e:
        logger.exception(f"Error while Ingestion of the Data from source", extra = properties)
        raise e 
        
    return tableDfWrite,countWrite

### Data quality

In [0]:
# username = dbutils.secrets.get(scope="adptst-kv-scope",key="adpBullhornSqlUserName")
# password = dbutils.secrets.get(scope="adptst-kv-scope",key="adpBullhornSqlUserPassword")
# jdbcHostname="10.5.16.73"
# driver = "com.microsoft.sqlserver.jdbc.SQLServerDriver"
# jdbcPort=1433
# jdbcDatabase= "DM_KellyServicesEMS"
# jdbcUrl = f"jdbc:sqlserver://{jdbcHostname}:{jdbcPort};database={jdbcDatabase};instanceName=BHDM_SQA"

# connectionProperties = { "user" : username, "password" : password, "driver" : driver}

#### readSparkCsv Function

In [0]:
def readSparkCsv(path):
    dfCsv = spark.read.csv(path, inferSchema = True, header = True)
    return dfCsv

#### sendEmailAlertDQ function

In [0]:
def sendEmailAlertDQ(senderEmail, recipientEmail, emailSubject, BodyOfTheMail):
    try:
        # Set up an SMTP connection
        server = smtplib.SMTP_SSL('smtp.sendgrid.net', 465)
        server.ehlo()

        # Log in using the sender's email and password
        server.login(apiKey, apiSecret)
        
        # Compose the email message
        msg = MIMEMultipart()
        msg['From'] = senderEmail
        msg['To'] = recipientEmail
        msg['Subject'] = emailSubject

        # Attach the body of the email as a MIMEText object
        text = MIMEText(BodyOfTheMail,'plain')
        msg.attach(text)
        
        # Send the email
        server.sendmail(senderEmail, recipientEmail, msg.as_string())

        # Close the SMTP connection
        server.quit()
        logger.info("Email alert sent successfully!",extra = properties)
    except Exception as e:
        logger.error(f"Error sending email alert: as {e}",extra = properties)

#### 1. masterRules function

In [0]:
def masterRules(auditAdbPipelineID, sourceSystem, sourceObjectName, targetObjectName, columnName, ruleName, ruleID, mappingID, criticality, sourceRuleQuery, targetRuleQuery, dateColumnName, primaryKey, lastRunDate, currentRunDate):
    try:
        logger.info("masterRules function definition has started", extra = properties)
        if int(ruleID) == 1:
            countCheck(auditAdbPipelineID, sourceSystem, sourceObjectName, targetObjectName, columnName, ruleName, ruleID, mappingID, criticality, dateColumnName, primaryKey, currentRunDate)
        elif int(ruleID) == 2:
            dqMd5Check(auditAdbPipelineID, sourceSystem, sourceObjectName, targetObjectName, columnName, ruleName, ruleID, mappingID, criticality,dateColumnName,primaryKey,lastRunDate, currentRunDate)
        elif int(ruleID) == 3:
            customQuery(auditAdbPipelineID, sourceSystem, sourceObjectName, targetObjectName, columnName, ruleName, ruleID, mappingID, criticality, sourceRuleQuery, targetRuleQuery)
        else:
            logger.info(f"No ruleID present for {sourceObjectName}", extra = properties)
        logger.info("masterRules function definition has completed successfully", extra = properties)
        
    except Exception as e:
        logger.exception(f"masterRules function definition failed with error as {e}", extra = properties)
        raise e 

#### 2. dqResult Function

In [0]:
def dqResultFunction(auditAdbPipelineID, ruleID, mappingID, ruleName, expectedResult, actualResult, dqStatus, summary, columnName, dqStartTs, dqEndTs, alertStatus):
    try:
        logger.info(f"dqResult Function started for {sourceObjectName}", extra = properties)
        schema = StructType([StructField("adbPipelineID", StringType(), True) \
                         ,StructField("ruleID", IntegerType(), True) \
                         ,StructField("mappingID", IntegerType(), True) \
                         ,StructField("ruleName", StringType(), True) \
                         ,StructField("expectedResult", StringType(), True) \
                         ,StructField("actualResult", StringType(), True) \
                         ,StructField("dqStatus", StringType(), True) \
                         ,StructField("summary", StringType(), True) \
                         ,StructField("columnName", StringType(), True) \
                         ,StructField("dqStartTs", TimestampType(), True) \
                         ,StructField("dqEndTs", TimestampType(), True) \
                         ,StructField("alertStatus", StringType(), True)])
        
        newRow = spark.createDataFrame([(auditAdbPipelineID, ruleID, mappingID, ruleName, expectedResult, actualResult, dqStatus, summary, columnName, dqStartTs, dqEndTs, alertStatus)], schema = schema)
        newRow.write.option("mergeSchema","True").mode("append").saveAsTable(dataQualityTable)
        logger.info(f"dqResult Function completed and result is written to {dataQualityTable}", extra = properties)
    except Exception as e:
        logger.exception(f" dqResult failed with the error as {e}", extra = properties)
        raise e

#### 3. countCheck function

In [0]:
def countCheck(auditAdbPipelineID, sourceSystem, sourceObjectName, targetObjectName, columnName, ruleName, ruleID, mappingID, criticality, dateColumnName, primaryKey, currentRunDate):
  try:
    logger.info(f"Count check function started for {sourceObjectName}", extra = properties)
    dqStartTs = datetime.now()
    alertStatus = "No"
    
    if dateColumnName != None and primaryKey != None:
      queryCount = f"SELECT COUNT(*) AS count FROM {sourceObjectName} WHERE {dateColumnName}  < convert(date,'{currentRunDate}')"
      sourceCount = spark.read.format("com.microsoft.sqlserver.jdbc.spark") \
                                               .option("driver", driver) \
                                               .option("url", jdbcUrl) \
                                               .option("user", username) \
                                               .option("password", password) \
                                               .option("query", queryCount) \
                                               .option("trustServerCertificate", "true").load().collect()[0][0]
      targetCount = spark.read.format("delta") \
                                    .load(deltaPath) \
                                    .filter((col("dwIsActive") == "A") | (col("dwIsActive") == "D")).count()
    else:
      queryCount = f"SELECT COUNT(*) AS count FROM {sourceObjectName}" 
      sourceCount = spark.read.format("com.microsoft.sqlserver.jdbc.spark") \
                                               .option("driver", driver) \
                                               .option("url", jdbcUrl) \
                                               .option("user", username) \
                                               .option("password", password) \
                                               .option("query", queryCount) \
                                               .option("trustServerCertificate", "true").load().collect()[0][0]
      targetCount = spark.read.format("delta").load(deltaPath).count()
      
    
    if int(sourceCount) == int(targetCount): 
      dqStatus = "Success"
      summary = "Source and Target count matched using Count Check function"
    else:
      dqStatus = "Failure"
      summary = "Source and Target count did not match using Count Check function"
      if criticality.strip().lower() == 'high':
        emailSubject= f"ALERT : {ruleName} Data Quality Rule failure for {sourceObjectName}"
        BodyOfTheMail="""
        Hi,
        DQ check failure encountered. Following are the details:
        adbPipelineID : """ + auditAdbPipelineID +""",
        sourceSystem : """+ sourceSystem +""",
        sourceObjectName : """+ sourceObjectName +""",
        targetObjectName : """+ targetObjectName +""",
        ruleName : """+ ruleName +""",
        ruleID : """+ str(ruleID) +""",
        mappingID : """+ str(mappingID) +""",
        sourceCount : """+ str(sourceCount) +""",
        targetCount : """+ str(targetCount) +"""
        
        Regards,
        ACE Data Platform"""
        
        sendEmailAlertDQ(senderEmail, recipientEmail,emailSubject,BodyOfTheMail)
        logger.info("Email Alerts has been sent for Criticality level 'high' and dqStatus 'failure' for dataQualityResult table ",extra=properties)
        alertStatus = "Yes"
        logger.info("Email Alerts has been triggered successfully and updated the flag of alertStatus to Yes ",extra=properties)
          
    dqEndTs = datetime.now()
    dqResultFunction(auditAdbPipelineID, ruleID, mappingID, ruleName, sourceCount, targetCount, dqStatus, summary, columnName, dqStartTs, dqEndTs, alertStatus)
    logger.info(f"""Count check captured successfully for {sourceObjectName}""", extra = properties)
    
  except Exception as e:
    logger.exception(f"Count check failed for {sourceObjectName} with error as {e}", extra = properties)
    raise e

#### 4. dqMd5Check function

In [0]:
def dqMd5Check(auditAdbPipelineID, sourceSystem, sourceObjectName, targetObjectName, columnName, ruleName, ruleID, mappingID, criticality, dateColumnName, primaryKey, lastRunDate, currentRunDate):
  try:
    logger.info(f"dqMd5Check function called for {sourceObjectName}", extra = properties)
    dqStartTs = datetime.now()
    alertStatus = "No"
    failureList = []
    failureListString = json.dumps(failureList)
    
    if dateColumnName != None and primaryKey != None:
      pkList = primaryKey.split(",")
      pkList = [element.strip() for element in pkList]
      query = f"SELECT * FROM {sourceObjectName} WHERE {dateColumnName} >= '{lastRunDate.strftime('%Y-%m-%d %H:%M:%S')}' AND {dateColumnName} < '{currentRunDate.strftime('%Y-%m-%d %H:%M:%S')}'"
      dfSource = spark.read.format("com.microsoft.sqlserver.jdbc.spark") \
                                               .option("driver", driver) \
                                               .option("url", jdbcUrl) \
                                               .option("user", username) \
                                               .option("password", password) \
                                               .option("query", query) \
                                               .option("trustServerCertificate", "true").load()
      dfSourceSorted = dfSource.select(sorted(dfSource.columns,reverse=False))
      dfSourceMd5 = generateMd5(dfSourceSorted)
      dfSourceRenamed = dfSourceMd5.withColumnRenamed("dwMd5CheckSum","md5CheckSumSource")
      dfSourceCheck = dfSourceRenamed.select(*pkList, "md5CheckSumSource")  
      
      dfTarget = spark.read.format("delta").load(deltaPath)
      dfTargetCheck = dfTarget.select(*pkList, dateColumnName,"dwMd5CheckSum") \
                              .filter((col(dateColumnName) >= lastRunDate.strftime('%Y-%m-%d %H:%M:%S')) \
                                                & (col(dateColumnName) < currentRunDate.strftime('%Y-%m-%d %H:%M:%S')))
      # to get the max dateLastSync for same primary key
      w = Window.partitionBy(*pkList)
      dfMax = dfTargetCheck.withColumn("rank", rank().over(w.orderBy(dfTargetCheck[dateColumnName].desc())))
      dfMaxTarget = dfMax.filter(dfMax["rank"] == 1)
      
      # inner join based on primary key
      dfJoin = dfSourceCheck.join(dfMaxTarget, pkList, "inner")
      dfJoinCheck = dfJoin.withColumn("isSame", when(dfJoin.md5CheckSumSource == dfJoin.dwMd5CheckSum, True).otherwise(False))
      if dfJoinCheck.filter(col("isSame") == False).count() == 0:
        dqStatus = 'Success'
        summary = 'Source and Target md5CheckSum value matched using dqMd5Check function'
      else:
        #to pass list of values for failed records
        dfJoinedFailed = dfJoinCheck.select(*pkList).filter(col("isSame") == False)
        failureList = []
        for row in dfJoinedFailed.collect():
          forEachList = []
          for j in range(0,len(dfJoinedFailed.columns)):
            forEachList.append(row[j])
          failureList.append(forEachList)
        failureListString = json.dumps(failureList)
        
        dqStatus = 'Failure'
        summary = f'Source and Target md5CheckSum value did not match using dqMd5Check function with primary keys as {failureListString}'
        
        logger.info(f"Failure happened for the primary keys details as {failureListString}")
        if criticality.strip().lower() == 'high':
          emailSubject= f"ALERT : {ruleName} Data Quality Rule failure for {sourceObjectName}"
          BodyOfTheMail="""
          Hi,
          DQ check failure encountered and below are the details:"
          adbPipelineID : """ + auditAdbPipelineID +""",
          sourceSystem : """+ sourceSystem +""",
          sourceObjectName : """+ sourceObjectName +""",
          targetObjectName : """+ targetObjectName +""",
          ruleName : """+ ruleName +""",
          ruleID : """+ str(ruleID) +""",
          mappingID : """+ str(mappingID) +""",
          primaryKeyList : """+ str(failureListString) +"""
          
          Regards,
          ACE Data Platform""" 
          
          sendEmailAlertDQ(senderEmail, recipientEmail, emailSubject, BodyOfTheMail)
          logger.info("Email Alerts has been sent for Criticality level 'high' and dqStatus 'failure' ",extra=properties)
          alertStatus = "Yes"
          logger.info("Email Alerts has been triggered successfully and updated the flag of alertStatus to Yes ",extra=properties)
    else:
      # Full load
      query = f"SELECT * FROM {sourceObjectName}"
      dfSource = spark.read.format("com.microsoft.sqlserver.jdbc.spark") \
                                                 .option("driver", driver) \
                                                 .option("url", jdbcUrl) \
                                                 .option("user", username) \
                                                 .option("password", password) \
                                                 .option("query", query) \
                                                 .option("trustServerCertificate", "true").load()
      dfSourceSorted = dfSource.select(sorted(dfSource.columns,reverse=False))
      dfSourceMd5 = generateMd5(dfSourceSorted)
      dfSourceRenamed = dfSourceMd5.withColumnRenamed("dwMd5CheckSum","md5CheckSumSource")
      dfSourceCheck = dfSourceRenamed.select("md5CheckSumSource")
      
      dfTarget = spark.read.format("delta").load(deltaPath)
      dfTargetCheck = dfTarget.select("dwMd5CheckSum")
      diff = dfSourceCheck.exceptAll(dfTargetCheck)
      if diff.count() == 0:
        dqStatus = "Success"
        summary = 'Source and Target md5CheckSum value matched using dqMd5Check function'
      else:
        dqStatus = 'Failure'
        summary = 'Source and Target md5CheckSum value did not match using dqMd5Check function'
        if criticality.strip().lower() == 'high':
          user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().tags().apply('user')
          emailSubject= f"ALERT : {ruleName}: Data Quality Rule failure for {sourceObjectName}"
          BodyOfTheMail="""
          Hi,
          DQ check failure encountered. Following are the details:"
          adbPipelineID : """ + auditAdbPipelineID +""",
          sourceSystem : """+ sourceSystem +""",
          sourceObjectName : """+ sourceObjectName +""",
          targetObjectName : """+ targetObjectName +""",
          ruleName : """+ ruleName +""",
          ruleID : """+ str(ruleID) +""",
          mappingID : """+ str(mappingID) +""",
          sourceObjectName : """+ str(sourceObjectName) +""",
          targetObjectName : """+ str(targetObjectName) +""" 
          
          Regards,
          ACE Data Platform"""
          
          sendEmailAlertDQ(senderEmail, recipientEmail,emailSubject,BodyOfTheMail)
          logger.info("Email Alerts has been sent for Criticality level 'high' and dqStatus 'failure' ",extra=properties)
          alertStatus = "Yes"
          logger.info("Email Alerts has been triggered successfully and updated the flag of alertStatus to Yes ",extra=properties) 
    
    dqEndTs = datetime.now()
    dqResultFunction(auditAdbPipelineID, ruleID, mappingID, ruleName, "null", "null", dqStatus, summary, columnName, dqStartTs, dqEndTs, alertStatus)
    logger.info(f"dqMd5Check function completed for {sourceObjectName}", extra = properties)
  except Exception as e:
    logger.exception(f"dqMd5Check function failed for {sourceObjectName} with the error as : {e}", extra = properties)
    raise e

#### 5. customQuery Function

In [0]:
def customQuery(auditAdbPipelineID, sourceSystem, sourceObjectName, targetObjectName, columnName, ruleName, ruleID, mappingID, criticality, sourceRuleQuery, targetRuleQuery):
  try:
        logger.info("Custom query function called", extra = properties)
        dqStartTs = datetime.now()
        alertStatus = 'No'
        
        sourceQuery = spark.read.format("com.microsoft.sqlserver.jdbc.spark") \
                                         .option("driver", driver) \
                                         .option("url", jdbcUrl) \
                                         .option("user", username) \
                                         .option("password", password) \
                                         .option("query", sourceRuleQuery) \
                                         .option("trustServerCertificate", "true").load()
        sourceQueryVal = sourceQuery.collect()[0][0]
        targetQueryVal = spark.sql(targetRuleQuery).collect()[0][0]
        
        if sourceQueryVal == targetQueryVal:        
            dqStatus = "Success"
            summary = "Source and Target results matched using Custom query check"
            
        else:
            dqStatus = "Failure"
            summary = "Source and Target results did not match using Custom query check"
            if criticality.strip().lower() == 'high':
                user = dbutils.notebook.entry_point.getDbutils().notebook().getContext().tags().apply('user')
                emailSubject= f"ALERT : {ruleName}: Data Quality Rule failure for {sourceObjectName}"
                BodyOfTheMail="""
                Hi,
                DQ check failure encountered. Following are the details:"
                adbPipelineID : """ + auditAdbPipelineID +""",
                sourceSystem : """+ sourceSystem +""",
                sourceObjectName : """+ sourceObjectName +""",
                targetObjectName : """+ targetObjectName +""",
                ruleName : """+ ruleName +""",
                ruleID : """+ str(ruleID) +""",
                mappingID : """+ str(mappingID) +""",
                sourceRuleOutput : """+ str(sourceQueryVal) +""",
                targetRuleOutput : """+ str(targetQueryVal) +"""
                
                Regards,
                ACE Data Platform"""
                
                sendEmailAlertDQ(senderEmail, recipientEmail,emailSubject,BodyOfTheMail)
                logger.info("Email Alerts has been sent for Criticality level 'high' and dqStatus 'failure' for dataQualityResult table ",extra=properties)
                alertStatus = "Yes"
                logger.info("Email Alerts has been triggered successfully and updated the flag of alertStatus to Yes ",extra=properties) 
            
        dqEndTs = datetime.now()
        dqResultFunction(auditAdbPipelineID, ruleID, mappingID, ruleName, sourceQueryVal, targetQueryVal, dqStatus, summary, columnName, dqStartTs, dqEndTs, alertStatus)
        logger.info("Custom query function ran successfully", extra = properties)
  except Exception as e:
    logger.exception(f"Custom query failed for {sourceObjectName} with error as {e}", extra = properties)
    raise e